In [2]:
import os, json, statistics
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from src import Questions
import matplotlib.patches as mpatches
from scipy.stats import ttest_ind
from itertools import combinations

In [3]:
# definitions
models = [
    "gpt_4o",
    "llama31_70b_Inst",
    "llama31_8B_Inst",
    "llama32_1B_Inst",
    "qwen25_72b_Inst",
    "Qwen25_7B_Inst",
    "Qwen25_1dot5B_Inst"
]

demography_profiles = [
    "employment_sector",
    "race",
    "education",
    "sex",
    "age",
    "language",
    "nationality_group"
]
SCORED_ANNOTATOR_DIR = "annotation_artefacts/final_annotator_objects"
EXCLUDED_SCORES = ["2_13", "23_", "24_", "25_", "27_"]
EXCLUDED_WHOLE = [23, 24, 25, 27]
questions = Questions().get_questions()

task_group_dict = {}
for i, k in enumerate(questions.keys()):
    qtype = questions[k]["question_type"]
    if qtype in task_group_dict:
        task_group_dict[qtype].append(i)
    else:
        task_group_dict[qtype] = [i]

stacked_model_df = []
for model in models:
    model_specific_scores = []
    for annfilename in os.listdir(SCORED_ANNOTATOR_DIR):
        annobj_for_df = {}
        annotator_filepath = os.path.join(SCORED_ANNOTATOR_DIR, annfilename)
        with open(annotator_filepath, "r") as ao:
            annotator_object = json.load(ao)
        
        annobj_for_df["id"] = annotator_object["id"]
        annobj_for_df.update(**annotator_object["profile"])
        
        for i, (question, scores_for_question) in enumerate(annotator_object["answers"].items()):
            if not scores_for_question["answer"]:
                continue
            else:
                for qdi, scores in scores_for_question["downstream_scores"][model].items():
                    annobj_for_df[qdi] = list(scores.values())[0]
        model_specific_scores.append(annobj_for_df)
    score_df = pd.DataFrame(model_specific_scores)
    exclude_cols = ["id"] + demography_profiles
    task_cols = [col for col in score_df.columns if col not in exclude_cols]
    stacked_model_df.append(score_df[task_cols])

def get_min_max_from_stacked_df(stacked_model_df):
    # Stack all DataFrames into one
    combined_df = pd.concat(stacked_model_df, ignore_index=True)

    # Compute min and max for each column
    return {
        col: {
            'min': combined_df[col].min(),
            'max': combined_df[col].max()
        }
        for col in combined_df.columns
    }

universal_scoredf_minmax = get_min_max_from_stacked_df(stacked_model_df)

scorelists_by_model = {}
for model in models:
    model_specific_scores = []
    for annfilename in os.listdir(SCORED_ANNOTATOR_DIR):
        annobj_for_df = {}
        annotator_filepath = os.path.join(SCORED_ANNOTATOR_DIR, annfilename)
        with open(annotator_filepath, "r") as ao:
            annotator_object = json.load(ao)
        
        annobj_for_df["id"] = annotator_object["id"]
        annobj_for_df.update(**annotator_object["profile"])
        
        for i, (question, scores_for_question) in enumerate(annotator_object["answers"].items()):
            if not scores_for_question["answer"]:
                continue
            else:
                for qdi, scores in scores_for_question["downstream_scores"][model].items():
                    annobj_for_df[qdi] = list(scores.values())[0]
        model_specific_scores.append(annobj_for_df)
    score_df = pd.DataFrame(model_specific_scores)
    exclude_cols = ["id"] + demography_profiles
    task_cols = [col for col in score_df.columns if col not in exclude_cols]

    fullscores_per_dp = {}
    for dp in demography_profiles:
        fullscores_per_dp[dp] = {}
        for tc in task_cols:
            fullscores_per_dp[dp][tc] = {}
            if not any(tc.startswith(prefix) for prefix in EXCLUDED_SCORES):
                demo_grouped = score_df.groupby([dp])
                for demo_name, group in demo_grouped:
                    # implement here for exceptions
                    if dp == "employment_sector":
                        group = group[group["language"] == "L1"]
                    elif dp == "education":
                        group = group[group["language"] == "L1"]
                    elif dp == "sex":
                        group = group[group["language"] == "L1"]
                        
                    elif(demo_name == "Black"):
                        group = group[group["language"] == "L1"]
                    elif(demo_name == "White"):
                        group = group[group["language"] == "L1"]
                    # implement here for exceptions
                    fullscores_per_dp[dp][tc][demo_name[0]] = group[tc].dropna().to_list()
    scorelists_by_model[model] = fullscores_per_dp

In [4]:
print(scorelists_by_model['gpt_4o'].keys())
print(scorelists_by_model['gpt_4o']['employment_sector'].keys())
print(scorelists_by_model['gpt_4o']['employment_sector']['0_0'].keys())
print(scorelists_by_model['gpt_4o']['employment_sector']['1_0'].keys())
print(scorelists_by_model['gpt_4o']['employment_sector']['1_0'].values())

dict_keys(['employment_sector', 'race', 'education', 'sex', 'age', 'language', 'nationality_group'])
dict_keys(['0_0', '1_0', '2_11', '2_3', '2_4', '2_10', '2_5', '2_2', '2_1', '2_6', '2_8', '2_14', '2_13', '2_9', '2_7', '2_0', '2_12', '3_0', '4_0', '6_0', '8_0', '9_0', '11_0', '12_0', '13_0', '14_0', '15_0', '16_0', '17_0', '18_0', '19_0', '20_0', '21_0', '22_0', '26_0', '26_1', '5_0', '7_0', '10_0', '28_0'])
dict_keys(['Engineering-STEM', 'Health-STEM', 'Humanities'])
dict_keys(['Engineering-STEM', 'Health-STEM', 'Humanities'])
dict_values([[0.9137484905985856, 0.9236874236874236, 0.8915558126084442, 0.9177601887102792, 0.9077505575075002, 0.9238525785921544, 0.9499276955924356, 0.935897435897436, 0.933806981157054, 0.9519377112738108, 0.9338833702651476, 0.911682055399438, 0.923760914227016, 0.9318429661941112, 0.9297637251714768, 0.937844611528822], [0.9076676033721396, 0.901792192278862, 0.8552496139989707, 0.9216990109982774, 0.9178814207715942, 0.939952923091704, 0.9339236156997

In [5]:
import math
from scipy.stats import f_oneway

def get_minmax_vals(data_dict):
    all_values = [x for group in data_dict.values() for x in group]
    return min(all_values), max(all_values)

def compute_diversity_score(data_dict):
    if len(data_dict) < 2:
        return 0  # No diversity if less than 2 elements
    
    min_val, max_val = get_minmax_vals(data_dict)

    list_of_values = []
    for lv in data_dict.values():
        if max_val == min_val:
            # All values are the same → treat as constant
            avg = 0.5  # or any neutral default
        else:
            normalized = [(x - min_val) / (max_val - min_val) for x in lv]
            avg = sum(normalized) / len(normalized)
        list_of_values.append(avg)
    
    total = []
    for i in range(len(list_of_values)):
        for j in range(i + 1, len(list_of_values)):
            total.append(abs(list_of_values[i] - list_of_values[j]))

    return sum(total) / len(total)

def compute_pairwise_t_test(data_dict):
    results = {}
    for (key1, values1), (key2, values2) in combinations(data_dict.items(), 2):
        stat, p = ttest_ind(values1, values2, equal_var=False)
        results[(key1, key2)] = p
    return results

def compute_anova_score(data_dict):
    # normalizer
    min_val, max_val = get_minmax_vals(data_dict)
    
    if max_val == min_val:
        # All values are the same across all groups
        return float("nan"), float("nan")

    groups = []
    for scores in data_dict.values():
        norm = [(x - min_val) / (max_val - min_val) for x in scores]
        groups.append(norm)
    
    # Perform one-way ANOVA
    f_statistic, p_value = f_oneway(*groups)
    return f_statistic, p_value


In [29]:
P_THRESHOLD = 0.05

divscores_model_v_taskg = {"Model" : models}
pairwise_ttest_model_v_taskg = {"Model" : models}
anova_model_v_taskg = {"Model" : models}

for tg_name, tg_idxs in task_group_dict.items():
    divscore_obj_list = []
    pairwise_ttest_obj_list = []
    anova_obj_list = []
    for modl in models:
        disvscore_table_row = {"model" : modl}
        pairwise_ttest_table_row = {"model" : modl}
        anova_table_row = {"model" : modl}

        for k in scorelists_by_model[modl].keys():
            tasks = [qdi for qdi in scorelists_by_model[modl][k] if not any(qdi.startswith(prefix) for prefix in EXCLUDED_SCORES)]
            tasks = [qdi for qdi in tasks if int(qdi.split("_")[0]) in tg_idxs]
            
            divsc_total = 0
            pairwise_t_test_significant = []
            anova_significant = []
            
            for qdi in tasks:
                divsc_total += compute_diversity_score(scorelists_by_model[modl][k][qdi])
                
                pairwise_ttest = compute_pairwise_t_test(scorelists_by_model[modl][k][qdi])
                if any(v < P_THRESHOLD for v in pairwise_ttest.values()):
                    pairwise_t_test_significant.append(qdi)
                
                anova_f, anova_p = compute_anova_score(scorelists_by_model[modl][k][qdi])
                if anova_p < P_THRESHOLD:
                    anova_significant.append(qdi)

            disvscore_table_row[k] = divsc_total/len(tasks)
            pairwise_ttest_table_row[k] = {
                "rejectable" : pairwise_t_test_significant,
                "all_tasks" : tasks,
            }
            anova_table_row[k] = {
                "rejectable" : anova_significant,
                "all_tasks" : tasks,
            }
        
        divscore_obj_list.append(disvscore_table_row)
        pairwise_ttest_obj_list.append(pairwise_ttest_table_row)
        anova_obj_list.append(anova_table_row)

    # ik bro but im paranoid
    assert(models == [m["model"] for m in divscore_obj_list])
    assert(models == [m["model"] for m in pairwise_ttest_obj_list])
    assert(models == [m["model"] for m in anova_obj_list])

    divscores_model_v_taskg[tg_name] = [statistics.mean([s for s in ls.values() if isinstance(s, float)]) for ls in divscore_obj_list]
    
    pairwise_ttest_model_v_taskg[tg_name] = []
    for ptt_by_model in pairwise_ttest_obj_list:
        all_rejectable = [v['rejectable'] if isinstance(v, dict) else [] for v in ptt_by_model.values()]
        all_rejectable = set([item for row in all_rejectable for item in row])
        pairwise_ttest_model_v_taskg[tg_name].append(len(all_rejectable)/len(ptt_by_model["employment_sector"]["all_tasks"]))
    
    anova_model_v_taskg[tg_name] = []
    for anova_by_model in anova_obj_list:
        all_rejectable = [v['rejectable'] if isinstance(v, dict) else [] for v in anova_by_model.values()]
        all_rejectable = set([item for row in all_rejectable for item in row])
        anova_model_v_taskg[tg_name].append(len(all_rejectable)/len(anova_by_model["employment_sector"]["all_tasks"]))

    def process_raw_obj(obj):
        return {k : (v if isinstance(v,str) else len(v["rejectable"])/len(v["all_tasks"])) for k, v in obj.items()}
    
    pairwise_ttest_obj_list_processed = [process_raw_obj(inst) for inst in pairwise_ttest_obj_list]
    anova_obj_list_processed = [process_raw_obj(inst) for inst in anova_obj_list]

    pd.DataFrame(divscore_obj_list).to_csv(f"score_tables/divscores/divscores_{tg_name}.csv")
    pd.DataFrame(pairwise_ttest_obj_list_processed).to_csv(f"score_tables/ptt/ptt_{tg_name}.csv")
    pd.DataFrame(anova_obj_list_processed).to_csv(f"score_tables/anova/anova_{tg_name}.csv")

pd.DataFrame(divscores_model_v_taskg).to_csv(f"score_tables/divscores/divscores_model_v_taskg.csv")
pd.DataFrame(pairwise_ttest_model_v_taskg).to_csv(f"score_tables/ptt/ptt_model_v_taskg.csv")
pd.DataFrame(anova_model_v_taskg).to_csv(f"score_tables/anova/anova_model_v_taskg.csv")

/Users/rayendito/miniconda3/envs/pietorch/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:573: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


In [29]:
scorelists_by_model[modl]

{'employment_sector': {'0_0': {'Engineering-STEM': [0.6295619474024918,
    0.577828917975364,
    0.6502331918906177,
    0.3499688639862553,
    0.4861847457847836,
    0.6115041044327875,
    0.5584851363492873,
    0.5599444527632863,
    0.5300322031217327,
    0.6176974680463052,
    0.5477317027851666,
    0.6110467304584837,
    0.6185807522371447,
    0.4998430412346591],
   'Health-STEM': [0.4306501926371851,
    0.4626804486989102,
    0.5928068462028985,
    0.5937423925667829,
    0.5858463536377173,
    0.5232577649171078,
    0.5948403762596619,
    0.6778105260383722,
    0.6762845028724147,
    0.6510191514761199,
    0.519996959730176,
    0.5203698977695167],
   'Humanities': [0.5229350925533153,
    0.6045145874217362,
    0.6622673599386415,
    0.6704176289673747,
    0.5509117004046298,
    0.6033267649837728,
    0.5989699834363512]},
  '1_0': {'Engineering-STEM': [0.911682055399438,
    0.8830946852425181,
    0.7576202098376688,
    0.7836902444300238,
    0.8

In [42]:
pd.DataFrame(divscore_obj_list)

,model,employment_sector,race,education,sex,age,language,nationality_group
0,gpt_4o,0.076681,0.060257,0.097152,0.064183,0.044605,0.052632,0.049019
1,llama31_70b_Inst,0.082223,0.069193,0.110523,0.055142,0.065924,0.081870,0.081134
2,llama31_8B_Inst,0.071448,0.052770,0.095833,0.096328,0.066579,0.083648,0.066716
3,llama32_1B_Inst,0.073708,0.061633,0.084720,0.064449,0.052513,0.063252,0.065317
4,qwen25_72b_Inst,0.080012,0.057944,0.106950,0.111769,0.052826,0.074510,0.065094
5,Qwen25_7B_Inst,0.093028,0.066860,0.088079,0.055605,0.055666,0.072879,0.075997
6,Qwen25_1dot5B_Inst,0.073091,0.049137,0.055796,0.065995,0.039495,0.063528,0.053146


In [43]:
# ANOVA P VALUE TESTS
REJECT_THRESHOLD = 0.05

ftest_obj_list = []
for modl in models:
    table_row = {"model" : modl}
    for k in scorelists_by_model[modl].keys():
        ftest_list=[]
        tasks = [qdi for qdi in scorelists_by_model[modl][k] if not any(qdi.startswith(prefix) for prefix in EXCLUDED_SCORES)]
        task_count = len(tasks)
        for qdi in tasks:
            f, p = compute_anova_score(scorelists_by_model[modl][k][qdi])
            ftest_list.append(f)
            adj_p = p * task_count * len(models)
            # adj_p = p * task_count
            if (adj_p < REJECT_THRESHOLD):
                print(modl, "REJECTED!", f"task-{qdi}", k, adj_p)
        ftest_filtered = [x for x in ftest_list if not math.isnan(x)]
        table_row[k] = sum(ftest_filtered) / len(ftest_filtered) if ftest_filtered else None
    ftest_obj_list.append(table_row)

ftest_df = pd.DataFrame(ftest_obj_list)
numeric_df = ftest_df.drop(columns=['model'])
row_means = numeric_df.mean(axis=1)
ftest_df['MODEL WISE MEAN'] = row_means

column_means = numeric_df.mean()
mean_row = column_means.copy()
mean_row['model'] = 'DEMOGRAPHY MEAN'
mean_row['MODEL WISE MEAN'] = float("nan")

df = pd.concat([ftest_df, pd.DataFrame([mean_row])], ignore_index=True)
df

llama32_1B_Inst REJECTED! task-21_0 language 0.04086319039134498


,model,employment_sector,race,education,sex,age,language,nationality_group,MODEL WISE MEAN
0,gpt_4o,0.938733,1.059700,0.997535,1.255356,1.232642,0.824975,1.154868,1.066258
1,llama31_70b_Inst,1.282116,1.790050,1.187866,0.573854,1.369606,1.539203,1.662821,1.343645
2,llama31_8B_Inst,0.867621,1.341729,1.174878,1.059561,1.416391,1.531551,1.844491,1.319460
3,llama32_1B_Inst,0.910052,1.405647,0.961751,0.711733,1.118108,2.003189,1.550193,1.237239
4,qwen25_72b_Inst,1.001818,1.043760,1.086093,1.218508,1.089574,1.291942,1.171789,1.129069
5,Qwen25_7B_Inst,1.180106,1.137426,1.382765,0.965277,1.434255,1.742754,1.026724,1.267044
6,Qwen25_1dot5B_Inst,1.167202,1.304088,0.796284,1.061088,0.968358,1.497051,1.395094,1.169881
7,DEMOGRAPHY MEAN,1.049664,1.297486,1.083881,0.977911,1.232705,1.490095,1.400854,NaN


In [68]:
def format_default(val):
    if isinstance(val, float):
        return f"{round(val, 2)}"
    return str(val)

def format_to_percentage(val):
    if isinstance(val, float):
        return f"{round(val * 100, 1)}\\%"
    return str(val)

def df_to_latex(df, format="default"):
    formatted_df = df.copy()
    for col in formatted_df.columns:
        if format == "default":
            formatfunc = format_default
        elif(format == "percentage"):
            formatfunc = format_to_percentage
        formatted_df[col] = formatted_df[col].apply(formatfunc)
    
    results_str = formatted_df.to_latex(index=False, escape=True)
    
    # patch correction:
    results_str = results_str.replace("70b", "70B").replace("72b", "72B").replace("qwen25", "Qwen25")
    results_str = results_str.replace(r"employment\_sector", "industry").replace(r"nationality\_group", "nationality")
    
    return results_str

def df_to_latex_one_column(df, caption, format="default"):
    return r"""\begin{table}[t]
\resizebox{0.49\textwidth}{!}{%
""" + df_to_latex(df, format) + rf"""}}
\caption{{{caption.replace("_",r"\_")}}}
\label{{tab:{caption}}}
\end{{table}}
"""
    pass

In [69]:
tgs = os.listdir("div_score_tables")

for tg in tgs:
    resdf = pd.read_csv(f"div_score_tables/{tg}", index_col=0)
    caption = tg.replace(".csv", "")
    print(df_to_latex_one_column(resdf, caption, format="default"))
    print("=================================")

\begin{table}[t]
\resizebox{0.49\textwidth}{!}{%
\begin{tabular}{llllllll}
\toprule
model & industry & race & education & sex & age & language & nationality \\
\midrule
gpt\_4o & 0.08 & 0.06 & 0.07 & 0.06 & 0.03 & 0.07 & 0.04 \\
llama31\_70B\_Inst & 0.1 & 0.06 & 0.06 & 0.03 & 0.03 & 0.07 & 0.05 \\
llama31\_8B\_Inst & 0.12 & 0.05 & 0.05 & 0.11 & 0.03 & 0.05 & 0.05 \\
llama32\_1B\_Inst & 0.1 & 0.08 & 0.05 & 0.04 & 0.05 & 0.07 & 0.07 \\
Qwen25\_72B\_Inst & 0.08 & 0.05 & 0.11 & 0.1 & 0.04 & 0.06 & 0.04 \\
Qwen25\_7B\_Inst & 0.14 & 0.06 & 0.09 & 0.08 & 0.04 & 0.07 & 0.04 \\
Qwen25\_1dot5B\_Inst & 0.1 & 0.07 & 0.1 & 0.09 & 0.07 & 0.12 & 0.07 \\
\bottomrule
\end{tabular}
}
\caption{divscores\_semantics}
\label{tab:divscores_semantics}
\end{table}

\begin{table}[t]
\resizebox{0.49\textwidth}{!}{%
\begin{tabular}{llllllll}
\toprule
model & industry & race & education & sex & age & language & nationality \\
\midrule
gpt\_4o & 0.07 & 0.03 & 0.1 & 0.05 & 0.04 & 0.03 & 0.04 \\
llama31\_70B\_Inst & 

In [7]:
questions

{'Sentiment refers to the emotional tone or attitude expressed in a piece of text, typically categorized as positive, negative,or neutral based on the conveyed feelings or opinions.<b>Example</b>"The address has been updated" has neutral sentiment.You are using a language model like ChatGPT to determine the sentiment of a text [TEXT] into one of possible <b>three sentiment labels: positive, negative, and neutral</b>. How would you phrase your request to the language model? <b>Your response must include the placeholder [TEXT].</b>': {'clean': 'Sentiment refers to the emotional tone or attitude expressed in a piece of text, typically categorized as positive, negative,or neutral based on the conveyed feelings or opinions.\n\nExample\n"The address has been updated" has neutral sentiment.\n\nYou are using a language model like ChatGPT to determine the sentiment of a text [TEXT] into one of possible three sentiment labels: positive, negative, and neutral.\n\nHow would you phrase your request

In [11]:
task_table = []
for i, q in enumerate(questions):
    for j in range(len(questions[q]["dataset"])):
        tc = f"{i}_{j}"
        task_obj = {
            "qdi" : tc,
            "Group" : questions[q]["question_type"]
        }
        if not any(tc.startswith(prefix) for prefix in EXCLUDED_SCORES):
            task_name = questions[q]["nickname"]
            dataset_name = questions[q]["dataset"][j]
            if "question_" not in dataset_name:
                task_name += f" - {dataset_name.replace('.csv', '')}"
                task_name = task_name.replace("500-","").replace("100-","").replace("grammatical_classification_","")
                # print(task_name)
            
            ds_df = pd.read_csv(f"question_artefacts/test_datasets/{dataset_name}")
            
            
            task_obj["Task Name and Benchmark"] = task_name
            task_obj["Dataset Size"] = len(ds_df)
            task_obj["Eval Metric"] = questions[q]["metrics"][0]
            
            task_table.append(task_obj)

task_table = pd.DataFrame(task_table)
print(task_table.to_latex(index=False, escape=True))

\begin{tabular}{lllrl}
\toprule
qdi & Group & Task Name and Benchmark & Dataset Size & Eval Metric \\
\midrule
0\_0 & semantics & Sentiment Analysis 3 Classes - sentiment\_3class & 500 & f1 \\
1\_0 & semantics & Sentiment Analysis 2 Classes - sentiment\_2class & 500 & f1 \\
2\_0 & machine\_translation & Machine Translation - eng\_Latn-zho\_Hans & 100 & bleu \\
2\_1 & machine\_translation & Machine Translation - eng\_Latn-rus\_Cyrl & 100 & bleu \\
2\_2 & machine\_translation & Machine Translation - eng\_Latn-spa\_Latn & 100 & bleu \\
2\_3 & machine\_translation & Machine Translation - eng\_Latn-deu\_Latn & 100 & bleu \\
2\_4 & machine\_translation & Machine Translation - eng\_Latn-jpn\_Jpan & 100 & bleu \\
2\_5 & machine\_translation & Machine Translation - eng\_Latn-ind\_Latn & 100 & bleu \\
2\_6 & machine\_translation & Machine Translation - eng\_Latn-tgl\_Latn & 100 & bleu \\
2\_7 & machine\_translation & Machine Translation - eng\_Latn-hin\_Deva & 100 & bleu \\
2\_8 & machine\_trans

In [12]:
SCORED_ANNOTATOR_DIR = "annotation_artefacts/final_annotator_objects"
question_valid_ds = []
for q in questions:
    counter = 0
    for annfilename in os.listdir(SCORED_ANNOTATOR_DIR):
        annotator_filepath = os.path.join(SCORED_ANNOTATOR_DIR, annfilename)
        with open(annotator_filepath, "r") as  ao:
            annotator_object = json.load(ao)
        if annotator_object["answers"][q]["answer"]:
            counter += 1
    question_valid_ds.append({
        "Task" : questions[q]["nickname"],
        "Valid answers" : counter,
    })
        
question_valid_ds_df = pd.DataFrame(question_valid_ds).drop(EXCLUDED_WHOLE)
print(question_valid_ds_df.to_latex(index=False, escape=True))


\begin{tabular}{lr}
\toprule
Task & Valid answers \\
\midrule
Sentiment Analysis 3 Classes & 80 \\
Sentiment Analysis 2 Classes & 90 \\
Machine Translation & 92 \\
Dialogue Summarization & 97 \\
Topic Classification & 90 \\
Free Generation: Balonku Song & 93 \\
Free Generation: G30SPKI & 93 \\
Free Generation: Hijrah & 80 \\
Free Generation: Adhan & 84 \\
Free Generation: Karak Kaliang & 94 \\
Free Generation: Giant Drum & 90 \\
Free Generation: Scientific Letter & 83 \\
Free Generation: School Life & 97 \\
Free Generation: Olympics Medallists & 96 \\
Free Generation: Hillary Clinton & 98 \\
Free Generation: MBZUAI & 98 \\
Free Generation: Grammar Correction & 98 \\
Free Generation: Grammar Correction CoT & 98 \\
Free Generation: Arithmetic & 99 \\
Free Generation: Shakespeare-Prompt Jailbreak & 97 \\
Multiple Choice Questions & 88 \\
Extractive QA & 91 \\
MultispanQA & 92 \\
Grammar Correctness Classification & 92 \\
Linguistically-conditioned Text Generation & 79 \\
\bottomrule
\end{